In [5]:
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import PhaseEstimation, Permutation
from qiskit.circuit.library.arithmetic import PiecewiseLinearPauliRotations
import numpy as np


In [6]:
def hhl_circuit(
    UA: QuantumCircuit,
    b_prep: QuantumCircuit,
    num_eig_qubits: int,
    lambda_min: float,
    lambda_max: float,
):
    n = b_prep.num_qubits

    system = QuantumRegister(n, "sys")
    eig = QuantumRegister(num_eig_qubits, "eig")
    anc = QuantumRegister(1, "anc")

    qc = QuantumCircuit(system, eig, anc)

    # |b⟩
    qc.compose(b_prep, system, inplace=True)

    # QPE
    qpe = PhaseEstimation(num_eig_qubits, UA)
    qc.append(qpe, eig[:] + system[:])

    # Eigenvalue inversion
    slopes = [lambda_min / lambda_max]
    offsets = [0.0]

    inv_rot = PiecewiseLinearPauliRotations(
        num_state_qubits=num_eig_qubits,
        slopes=slopes,
        offsets=offsets,
        basis="Y"
    )

    qc.append(inv_rot, eig[:] + anc[:])

    # Uncompute QPE
    qc.append(qpe.inverse(), eig[:] + system[:])

    return qc


# Intial assumption about the Discretized Grid

We have a grid, say 8x8, and each point starts by being marked as shown. We are finding the steam function at each point.

<div>
    <img src="./grid_structure.png" width="400"/>
</div>

Thus we get a system of linear equations for each cell in the grid with its 4 neighbors. This gives us a sparse matrix A.

We ignore the boundary conditions for now, assuming they are all 0 as per the problem. This gives us a 6x6 grid [ $ (n-2)*(n-2) $ grid].

If we set the matrix $A$ for the 6x6 grid, we get the following grid, with the index becoming $ i + 6j $:

$ A = \begin{bmatrix}
\psi_{1,1} & \psi_{1,2} & ... & \psi_{1,6} & \psi_{2,1} & \psi_{2,2} & ... & \psi_{6,6} \\
... & ... & ... & ... & ... & ... & ... & ... \\
\end{bmatrix} $

This ends up being a 36x36 matrix. However, it is sparse, and more importantly, it can be made into a block-encoding. We need $ log_2(36) = 6 $ qubits to represent the system. Each row equation can be encoded as a set of instructions, so 

$ A = 4I - S_x - S_y - S_x^{\dagger} - S_y^{\dagger} $


In [7]:
def build_laplacian_block_encoding(
    grid_size: int = 6,
    num_qubits: int = 6
) -> QuantumCircuit:
    """
    Build a block-encoded unitary UA representing the 2D Laplacian operator:
    A = 4I - S_x - S_y - S_x† - S_y†
    
    For a discretized grid, S_x and S_y are shift operators that map
    |i, j⟩ -> |i±1, j⟩ and |i, j⟩ -> |i, j±1⟩ respectively.
    
    Args:
        grid_size: Size of the interior grid (default 6x6)
        num_qubits: Number of qubits needed (log2(grid_size²))
    
    Returns:
        QuantumCircuit representing the block-encoded operator
    """
    
    qc = QuantumCircuit(num_qubits, name="UA_Laplacian")
    qc_sx = _build_shift_operator(grid_size, 1, 0, num_qubits, name="Sx")
    qc_sy = _build_shift_operator(grid_size, 0, 1, num_qubits, name="Sy")
    
    # The block-encoded form is normalized:
    # UA = (1/4) * (4I - Sx - Sy - Sx† - Sy†)
    # This ensures the eigenvalues are in [-1, 1] range for block encoding

    qc.append(qc_sx.to_gate(), range(num_qubits))
    qc.append(qc_sy.to_gate(), range(num_qubits))
    qc.append(qc_sx.inverse().to_gate(), range(num_qubits))
    qc.append(qc_sy.inverse().to_gate(), range(num_qubits))

    return qc


def _build_shift_operator(
    grid_size: int,
    shift_i: int,
    shift_j: int,
    num_qubits: int,
    name: str = "Shift"
) -> QuantumCircuit:
    """
    Build a shift operator S that maps |i, j⟩ -> |i+shift_i, j+shift_j⟩ (with periodic BC)
    
    Args:
        grid_size: Grid dimension (grid_size x grid_size)
        shift_i: Shift in i direction
        shift_j: Shift in j direction
        num_qubits: Total number of qubits
        name: Name of the operator
    
    Returns:
        QuantumCircuit representing the shift operator
    """
    qc = QuantumCircuit(num_qubits, name=name)

    perm_matrix = np.zeros((2**num_qubits, 2**num_qubits))
    for idx in range(2**num_qubits):
        i = idx % grid_size
        j = idx // grid_size
        new_i = (i + shift_i) % grid_size
        new_j = (j + shift_j) % grid_size
        new_idx = new_i + grid_size * new_j
        if new_idx < 2**num_qubits:
            perm_matrix[new_idx, idx] = 1.0

    perm_pattern = [int(np.argmax(perm_matrix[:, i])) for i in range(perm_matrix.shape[1])]
    perm_gate = Permutation(num_qubits, perm_pattern)
    qc.append(perm_gate, range(num_qubits))
    
    return qc